### Задание

1. Выбрать QA датасет (https://huggingface.co/datasets) — **SberQuAD** (пары вопрос–контекст лежат в `trainval.pkl` / `test.pkl`).
2. На выбранном датасете обучить модель https://huggingface.co/intfloat/multilingual-e5-small.
3. Переписать код под актуальную версию sentence-transformers (`SentenceTransformerTrainer` вместо `model.fit`, `datasets.Dataset` вместо `InputExample`).
4. Сравнить результаты (zero-shot E5 vs дообученная E5).

Версии: `sentence-transformers==6.2.0`, `transformers>=5`, `torchmetrics`.

In [ ]:
import pickle, random
import numpy as np
import torch
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
from torchmetrics.functional.retrieval import (
    retrieval_reciprocal_rank, retrieval_average_precision, retrieval_normalized_dcg, retrieval_recall,
)

SEED = 42

MODEL_NAME = 'intfloat/multilingual-e5-small'
QUERY_PREFIX, PASSAGE_PREFIX = 'query: ', 'passage: '

/tmp/ipykernel_2281972/1377296410.py:5: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
/tmp/ipykernel_2281972/1377296410.py:6: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction


cuda


In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(device)

## Данные

In [ ]:
with open('trainval.pkl', 'rb') as f:
    trainval = pickle.load(f)
with open('test.pkl', 'rb') as f:
    test = pickle.load(f)
print(len(trainval), len(test))

contexts = sorted({c for _, c, _ in trainval})
rng = random.Random(SEED)
rng.shuffle(contexts)
val_ctx = set(contexts[:len(contexts) // 10])

train_rows = [r for r in trainval if r[1] not in val_ctx]
val_rows = [r for r in trainval if r[1] in val_ctx]
print('train pairs:', len(train_rows), 'val pairs:', len(val_rows))

def to_dataset(rows):
    q, c, s = zip(*rows)
    return Dataset.from_dict({'sentence1': list(q), 'sentence2': list(c), 'score': [float(x) for x in s]})

train_ds = to_dataset(train_rows).shuffle(seed=SEED)
print(train_ds[0])

100728 47872
train pairs: 90578 val pairs: 10150


{'sentence1': 'Что показали современные работы, связанные с анализом последовательностей ДНК?', 'sentence2': 'Нет единого мнения относительно родственных связей между различными группами семенных растений. В течение второй половины XX века большинство учёных считало голосеменные группой, парафилетической по отношению к цветковым растениям, однако современные работы, связанные с анализом последовательностей ДНК, показали, что современные голосеменные всё же являются монофилетической группой. Некоторые ботаники, например, А. Б. Шипунов, предлагают считать эту группу отделом, а перечисленные выше шесть таксонов — классами.', 'score': 1.0}


## Метрики близости пар и ретривал-метрики

In [ ]:
def pair_evaluator(rows, name, n=None):
    rows = rows[:n] if n else rows
    q, c, s = zip(*rows)

    return EmbeddingSimilarityEvaluator(
        sentences1=[QUERY_PREFIX + x for x in q],
        sentences2=[PASSAGE_PREFIX + x for x in c],
        scores=list(s), main_similarity=SimilarityFunction.COSINE, name=name, batch_size=128, write_csv=False,
    )

val_evaluator = pair_evaluator(val_rows, 'val')

rng2 = random.Random(SEED)
test_sample = rng2.sample(test, 10000)

test_evaluator = pair_evaluator(test_sample, 'test')

seen, queries, docs = set(), [], []
for q, c, s in test:
    if s == 1.0 and c not in seen:
        seen.add(c); queries.append(q); docs.append(c)
n = len(docs)
print('retrieval queries/docs:', n)

@torch.inference_mode()
def retrieval_metrics(model):
    d = model.encode([PASSAGE_PREFIX + x for x in docs], convert_to_tensor=True, normalize_embeddings=True, batch_size=128)
    q = model.encode([QUERY_PREFIX + x for x in queries], convert_to_tensor=True, normalize_embeddings=True, batch_size=128)
    sims = (q @ d.T).float().cpu()
    r5, mrr, rmap, ndcg = [], [], [], []

    for i in range(n):
        target = torch.zeros(n, dtype=torch.bool); target[i] = True
        p = sims[i]
        r5.append(retrieval_recall(p, target, top_k=5).item())
        mrr.append(retrieval_reciprocal_rank(p, target, top_k=5).item())
        rmap.append(retrieval_average_precision(p, target, top_k=5).item())
        ndcg.append(retrieval_normalized_dcg(p, target, top_k=10).item())

    return {f'Recall@5': np.mean(r5), 'MRR@5': np.mean(mrr), 'MAP@5': np.mean(rmap), 'NDCG@10': np.mean(ndcg)}, sims

retrieval queries/docs: 4425


# Метрики без дообучения

In [4]:
base_model = SentenceTransformer(MODEL_NAME, device=device)
print(base_model)
base_pair = {'val': val_evaluator(base_model), 'test': test_evaluator(base_model)}
base_ret, base_sims = retrieval_metrics(base_model)
print(base_pair); print(base_ret)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)


{'val': {'val_pearson_cosine': 0.9148284765729705, 'val_spearman_cosine': 0.8630651558419515}, 'test': {'test_pearson_cosine': 0.9125394687080641, 'test_spearman_cosine': 0.8624728097963988}}
{'Recall@5': np.float64(0.9240677966101695), 'MRR@5': np.float64(0.8393596990364419), 'MAP@5': np.float64(0.8393596990364419), 'NDCG@10': np.float64(0.870124147490593)}


# Дообучение

In [ ]:
model = SentenceTransformer(MODEL_NAME, device=device)
loss = losses.CosineSimilarityLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir='qa/checkpoints',
    num_train_epochs=5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    warmup_steps=0.1,
    weight_decay=0.01,
    bf16=True,
    prompts={'sentence1': QUERY_PREFIX, 'sentence2': PASSAGE_PREFIX},
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_val_spearman_cosine',
    greater_is_better=True,
    logging_steps=100,
    seed=SEED,
    report_to='none',
)

trainer = SentenceTransformerTrainer(
    model=model, args=args, train_dataset=train_ds, loss=loss, evaluator=val_evaluator,
)
trainer.train()
model.save_pretrained('qa/results')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


Epoch,Training Loss,Validation Loss,Val Pearson Cosine,Val Spearman Cosine
1,0.053719,No log,0.911006,0.852831
2,0.047038,No log,0.912405,0.852483
3,0.038349,No log,0.912138,0.850993
4,0.035424,No log,0.911436,0.850406
5,0.032462,No log,0.910780,0.849763


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# Загрузка чекпоинта и рассчёт метрик

In [ ]:
finetuned = SentenceTransformer('qa/results', device=device)
ft_pair = {'val': val_evaluator(finetuned), 'test': test_evaluator(finetuned)}
ft_ret, ft_sims = retrieval_metrics(finetuned)
print(ft_pair)
print(ft_ret)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'val': {'val_pearson_cosine': 0.9110426632761394, 'val_spearman_cosine': 0.8528470101040398}, 'test': {'test_pearson_cosine': 0.9032503764273273, 'test_spearman_cosine': 0.8499530353869432}}
{'Recall@5': np.float64(0.6854237288135593), 'MRR@5': np.float64(0.5482259893383683), 'MAP@5': np.float64(0.5482259893383683), 'NDCG@10': np.float64(0.6053646453491038)}


# Вариант 2: MultipleNegativesRankingLoss

In [7]:
from sentence_transformers.training_args import BatchSamplers

pos_rows = [r for r in train_rows if r[2] == 1.0]
pos_ds = Dataset.from_dict({'anchor': [r[0] for r in pos_rows], 'positive': [r[1] for r in pos_rows]}).shuffle(seed=SEED)
print(len(pos_ds))

mnrl_model = SentenceTransformer(MODEL_NAME, device=device)
mnrl_args = SentenceTransformerTrainingArguments(
    output_dir='qa/checkpoints_mnrl',
    num_train_epochs=2,
    per_device_train_batch_size=128,
    learning_rate=2e-5,
    warmup_steps=0.1,
    weight_decay=0.01,
    bf16=True,
    prompts={'anchor': QUERY_PREFIX, 'positive': PASSAGE_PREFIX},
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy='no',
    logging_steps=50,
    seed=SEED,
    report_to='none',
)
mnrl_trainer = SentenceTransformerTrainer(
    model=mnrl_model, args=mnrl_args, train_dataset=pos_ds, loss=losses.MultipleNegativesRankingLoss(mnrl_model),
)
mnrl_trainer.train()
mnrl_model.save_pretrained('qa/results_mnrl')

/tmp/ipykernel_2281972/1495472721.py:1: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers


45289


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


Step,Training Loss
50,1.607036
100,0.238723
150,0.220287
200,0.191360
250,0.177331
300,0.194561
350,0.186347
400,0.162857
450,0.149180
500,0.155825


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
mnrl_pair = {'val': val_evaluator(mnrl_model), 'test': test_evaluator(mnrl_model)}
mnrl_ret, mnrl_sims = retrieval_metrics(mnrl_model)
print(mnrl_pair); print(mnrl_ret)

{'val': {'val_pearson_cosine': 0.9370831627251569, 'val_spearman_cosine': 0.8638775960500206}, 'test': {'test_pearson_cosine': 0.935502741687144, 'test_spearman_cosine': 0.8633125010640617}}
{'Recall@5': np.float64(0.9396610169491525), 'MRR@5': np.float64(0.8525235408275141), 'MAP@5': np.float64(0.8525235408275141), 'NDCG@10': np.float64(0.8828524050200727)}


# Сравнение

In [9]:
import pandas as pd
rows = {
    'Spearman cosine (val)': (base_pair['val']['val_spearman_cosine'], ft_pair['val']['val_spearman_cosine'], mnrl_pair['val']['val_spearman_cosine']),
    'Spearman cosine (test)': (base_pair['test']['test_spearman_cosine'], ft_pair['test']['test_spearman_cosine'], mnrl_pair['test']['test_spearman_cosine']),
}
for k in base_ret:
    rows[k + f' (test, {n} docs)'] = (base_ret[k], ft_ret[k], mnrl_ret[k])
df = pd.DataFrame(rows, index=['zero-shot', 'fine-tuned: CosineSimilarityLoss', 'fine-tuned: MNRL']).T
df.round(4)

,zero-shot,fine-tuned: CosineSimilarityLoss,fine-tuned: MNRL
Spearman cosine (val),0.8631,0.8528,0.8639
Spearman cosine (test),0.8625,0.8500,0.8633
"Recall@5 (test, 4425 docs)",0.9241,0.6854,0.9397
"MRR@5 (test, 4425 docs)",0.8394,0.5482,0.8525
"MAP@5 (test, 4425 docs)",0.8394,0.5482,0.8525
"NDCG@10 (test, 4425 docs)",0.8701,0.6054,0.8829


## Пример поиска

In [10]:
for name, sims in [('zero-shot', base_sims), ('CosineSimilarityLoss', ft_sims), ('MNRL', mnrl_sims)]:
    i = 0
    top = sims[i].topk(3)
    print(f'=== {name} === Question: {queries[i]}  (correct doc id: {i})')
    for rank, (idx, sc) in enumerate(zip(top.indices.tolist(), top.values.tolist()), 1):
        print(f'  {rank}. doc {idx} cos={sc:.3f} {"<-- correct" if idx == i else ""}\n     {docs[idx][:200]}')

=== zero-shot === Question: Какое содержание селена в морской воде?  (correct doc id: 0)
  1. doc 0 cos=0.883 <-- correct
     Содержание селена в земной коре — около 500 мг/т. Основные черты геохимии селена в земной коре определяются близостью его ионного радиуса к ионному радиусу серы. Селен образует 37 минералов, среди кот
  2. doc 75 cos=0.852 
     Селен в организме взаимодействует с витаминами, ферментами и биологическими мембранами, участвует в регуляции обмена веществ, в обмене жиров, белков и углеводов, а также в окислительно-восстановительн
  3. doc 2529 cos=0.846 
     Селен по характеру действия несколько напоминает мышьяк; обладает политропным действием с преимущественным поражением печени, почек и ЦНС. Металлический селен менее ядовит. Из неорганических соединени
=== CosineSimilarityLoss === Question: Какое содержание селена в морской воде?  (correct doc id: 0)
  1. doc 2737 cos=0.892 
     Хотя химический состав озёр остаётся относительно длительное время постоянным, в о